# 03. Exploratory Data Analysis (EDA)

**Project:** Retail Sales Forecasting & Analytics  
**Phase:** Phase 6 – Exploratory Data Analysis  
**Data Source:** `data/processed/superstore_cleaned.csv`  

### Purpose & Scope
This notebook conducts a rigorous, empirical exploratory investigation of the cleaned Superstore retail dataset.
All findings, metrics, and patterns identified here establish the foundation for subsequent feature engineering and time-series forecasting.

> **Guardrail Reminder:** This phase is strictly exploratory. No ML modeling, lag feature creation, or forecasting is performed.

## 1. Setup and Ingestion
We load libraries and ingest `data/processed/superstore_cleaned.csv`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import PROCESSED_DATA_FILE
from src.analytics import (
    calculate_core_kpis,
    aggregate_by_time,
    calculate_category_performance,
    calculate_subcategory_performance,
    calculate_regional_performance,
    calculate_discount_tiers
)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

# Load cleaned data
df = pd.read_csv(PROCESSED_DATA_FILE)
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

print(f"Cleaned dataset loaded: {df.shape[0]:,} rows by {df.shape[1]} columns.")
df.head(2)

## 2. Dataset Overview: Transactions vs Unique Orders
We distinguish line-item records from unique purchase orders.

In [ ]:
overview = {
    "Total Transactions (Rows)": len(df),
    "Unique Orders": df['Order ID'].nunique(),
    "Unique Customers": df['Customer ID'].nunique(),
    "Unique Products": df['Product ID'].nunique(),
    "Categories": df['Category'].nunique(),
    "Sub-Categories": df['Sub-Category'].nunique(),
    "Regions": df['Region'].nunique(),
    "States": df['State'].nunique(),
    "Cities": df['City'].nunique(),
    "Earliest Order Date": df['Order Date'].min().strftime('%Y-%m-%d'),
    "Latest Order Date": df['Order Date'].max().strftime('%Y-%m-%d'),
    "Unique Order Dates": df['Order Date'].nunique()
}
pd.Series(overview)

## 3. Core Business KPIs
Calculated at both transaction and aggregated order levels.

In [ ]:
kpis = calculate_core_kpis(df)
for k, v in kpis.items():
    print(f"{k.replace('_', ' ').title()}: {v}")

## 4. Time-Based Sales Analysis: Daily, Weekly, Monthly, Yearly
Examine temporal trends, compound growth, and seasonal peaks.

In [ ]:
# Monthly aggregation
monthly = aggregate_by_time(df, freq='ME')
print("Monthly Summary Head (first 5 months):")
print(monthly[['Sales', 'Profit', 'Orders', 'Profit_Margin_Pct']].head())

# Yearly aggregation
df['Year'] = df['Order Date'].dt.year
yearly = df.groupby('Year').agg(
    Sales=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
    Orders=('Order ID', 'nunique'),
    Quantity=('Quantity', 'sum')
)
yearly['Profit_Margin_%'] = (yearly['Profit'] / yearly['Sales'] * 100).round(2)
yearly['YoY_Growth_%'] = (yearly['Sales'].pct_change() * 100).round(2)
yearly

In [ ]:
# Monthly trend plot
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(monthly.index, monthly['Sales'], color='#1f77b4', linewidth=2, marker='o', label='Monthly Sales ($)')
ax1.set_ylabel('Total Sales ($)', color='#1f77b4', weight='bold')
ax1.tick_params(axis='y', labelcolor='#1f77b4')

ax2 = ax1.twinx()
ax2.plot(monthly.index, monthly['Profit'], color='#2ca02c', linewidth=2, linestyle='--', marker='s', label='Monthly Profit ($)')
ax2.set_ylabel('Total Profit ($)', color='#2ca02c', weight='bold')
ax2.tick_params(axis='y', labelcolor='#2ca02c')

for yr in [2014, 2015, 2016, 2017]:
    ax1.axvspan(pd.Timestamp(f'{yr}-09-01'), pd.Timestamp(f'{yr}-12-31'), color='orange', alpha=0.12)

plt.title('Monthly Sales & Profit Trends with Q4 Holiday Highlighting', weight='bold')
plt.show()

## 5. Seasonality Analysis
Assess Month-of-Year seasonal curves across all 4 years.

In [ ]:
df['Month'] = df['Order Date'].dt.month
monthly_piv = df.groupby(['Year', 'Month'])['Sales'].sum().unstack(level=0)

fig, ax = plt.subplots(figsize=(10, 5))
for yr in monthly_piv.columns:
    ax.plot(range(1, 13), monthly_piv[yr], marker='o', label=f'Year {yr}')

ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
ax.set_ylabel('Total Sales ($)', weight='bold')
ax.set_title('Year-over-Year Seasonal Invariant: Recurring Q4 Surge', weight='bold')
ax.legend()
plt.show()

## 6. Category & Sub-Category Performance
Comparing top-line sales versus net profitability.

In [ ]:
cat_perf = calculate_category_performance(df)
print("=== Category Performance ===")
print(cat_perf)

sub_perf = calculate_subcategory_performance(df)
print("\n=== Top 5 and Bottom 3 Sub-Categories by Profit ===")
print(pd.concat([sub_perf.head(5), sub_perf.tail(3)]))

## 7. Customer Segment Performance

In [ ]:
seg_agg = df.groupby('Segment').agg(
    Sales=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
    Orders=('Order ID', 'nunique'),
    Quantity=('Quantity', 'sum')
)
seg_agg['Profit_Margin_%'] = (seg_agg['Profit'] / seg_agg['Sales'] * 100).round(2)
seg_agg['AOV'] = (seg_agg['Sales'] / seg_agg['Orders']).round(2)
seg_agg

## 8. Geographic Performance
Evaluate revenue and profit by Region and top/bottom States.

In [ ]:
reg_perf = calculate_regional_performance(df)
print("=== Regional Performance ===")
print(reg_perf)

state_perf = df.groupby('State').agg({'Sales': 'sum', 'Profit': 'sum', 'Order ID': 'nunique'})
state_perf['Margin_%'] = (state_perf['Profit'] / state_perf['Sales'] * 100).round(2)

print("\nTop 5 Profitable States:")
print(state_perf.sort_values(by='Profit', ascending=False).head(5))

print("\nBottom 5 Unprofitable States (Loss Centers):")
print(state_perf.sort_values(by='Profit', ascending=True).head(5))

## 9. Discount vs Profitability Analysis
Investigate the empirical association between discount rate and net margin.

In [ ]:
disc_tiers = calculate_discount_tiers(df)
print("=== Performance Across Discount Tiers ===")
print(disc_tiers)

fig, ax = plt.subplots(figsize=(8, 4.5))
colors = ['#2ca02c' if m >= 0 else '#d62728' for m in disc_tiers['Profit_Margin_Pct']]
ax.bar(disc_tiers.index, disc_tiers['Profit_Margin_Pct'], color=colors, width=0.5)
ax.axhline(0, color='black', linewidth=1)
ax.set_ylabel('Profit Margin (%)', weight='bold')
ax.set_title('Profit Margin Erosion Across Discount Tiers (Severe Loss >20%)', weight='bold')
plt.xticks(rotation=15)
plt.show()

## 10. Shipping Mode Analysis
Evaluate lead times and performance by fulfillment mode.

In [ ]:
df['Shipping_Days'] = (df['Ship Date'] - df['Order Date']).dt.days
ship_agg = df.groupby('Ship Mode').agg(
    Transactions=('Sales', 'count'),
    Sales=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
    Mean_Lead_Days=('Shipping_Days', 'mean'),
    Min_Lead_Days=('Shipping_Days', 'min'),
    Max_Lead_Days=('Shipping_Days', 'max')
)
ship_agg['Profit_Margin_%'] = (ship_agg['Profit'] / ship_agg['Sales'] * 100).round(2)
ship_agg

## 11. Correlation Analysis & Outlier Inspection
Evaluate collinearity and inspect high-value orders.

In [ ]:
num_cols = ['Sales', 'Quantity', 'Discount', 'Profit']
corr = df[num_cols].corr()
print("=== Correlation Matrix ===")
print(corr.round(3))

print("\nTop 3 Highest-Sales Transactions:")
print(df.sort_values(by='Sales', ascending=False)[['Order ID', 'Product Name', 'Sales', 'Quantity', 'Profit']].head(3))

## 12. Forecasting Architecture Summary
Evidence-based forecasting design derived from EDA:
- **Target:** `Sales` aggregated at weekly (`W-SUN`) or daily (`D`) frequency.
- **Seasonality:** Strong annual wave (Q4 surge accounts for 51.6% of annual volume).
- **Validation Split:** 2014-2016 Training (75%, 156 weeks) and 2017 Testing (25%, 52 weeks).
- **Feature Families:** Lags (1, 2, 4, 52), Rolling Windows (4, 8, 12 weeks), Calendar variables (Month, Quarter, WeekOfYear).